# Faithfulness experiments (Colab)
**Runtime → Change runtime type → T4 GPU.** Run top to bottom. Results are written straight to Google Drive,
so if Colab disconnects, just re-run the same cells: finished work is reused, not recomputed.

In [ ]:
# 1. Get / update the code
import os
if not os.path.exists('/content/BTech-Project'):
    # private repo? use https://<TOKEN>@github.com/Aman14o5/BTech-Project.git
    !git clone -b faithfulness-pilot https://github.com/Aman14o5/BTech-Project.git /content/BTech-Project
else:
    !git -C /content/BTech-Project pull
%cd /content/BTech-Project
!pip -q install transformers sentencepiece accelerate

In [ ]:
# 2. Drive, checkpoint and where to save results
from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = '/content/drive/MyDrive/nt8'                       # folder with config.json + weights
RESULTS    = '/content/drive/MyDrive/btp_results'               # everything is saved here
RUN_DIR    = f'{RESULTS}/t5_nt8_test'                            # one folder per model/run
!ls "$MODEL_PATH" 

In [ ]:
# 3. Sanity check (should print contradiction + a sensible explanation)
import sys; sys.path.insert(0, 'faithfulness')
from predictors import T5Predictor
pred = T5Predictor(MODEL_PATH)
pairs = [("A man getting a tattoo on his back.", "A woman is getting a tattoo."),
         ("Two dogs run on the grass.", "Animals are outside."),
         ("A girl reads a book.", "The girl is reading her favourite novel.")]
full, fast = pred.predict(pairs), pred.predict_labels(pairs)
for f, g in zip(full, fast):
    print(f["label"], {k: round(v, 3) for k, v in f["probs"].items()}, "|", f["explanation"])
    assert f["label"] == g["label"] and all(abs(f["probs"][k] - g["probs"][k]) < 1e-3 for k in f["probs"]), "fast != full"
print("fast label-only scoring matches full generation")
del pred

## 4. Build the counterfactual dataset (model-independent, ~10 s)
Human-highlighted evidence in all 9,824 e-SNLI test pairs → removal / isolation variants with matched random controls.
Also validates the explanation-stance rules on the human explanations.

In [ ]:
!python faithfulness/build_dataset.py --split test --out_dir "$RESULTS/dataset"
!cat "$RESULTS/dataset/esnli_cf_gold_test_stats.json" | head -40

## 5. Run the experiment
Start with `--n 2000` (~15-25 min on a T4). When that works, remove `--n 2000` to run all 9,824 test pairs
into a new `RUN_DIR` (~1-1.5 h; resumable). Out of memory → add `--batch_size 16 --label_batch_size 64`.

In [ ]:
!python faithfulness/run_experiment.py --model_path "$MODEL_PATH" --n 2000 --out_dir "$RUN_DIR" 

In [ ]:
# 6. Figures + tables
!python faithfulness/plots.py --run_dir "$RUN_DIR"
from IPython.display import Image, display, Markdown
import glob
for f in sorted(glob.glob(f'{RUN_DIR}/figures/*.png')):
    display(Markdown(f'**{os.path.basename(f)}**')); display(Image(f))

In [ ]:
# 7. Table view of the main result (cited minus control, with 95% CI)
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
t = pd.read_csv(f'{RUN_DIR}/tables/contrasts.csv')
t[['contrast','group','n','flip_diff','flip_lo','flip_hi','drop_diff']].round(3)

In [ ]:
# 8. Examples to read: explanation argues for a different label than predicted
df = pd.read_csv(f'{RUN_DIR}/per_example.csv')
cols = ['gold','pred','p_pred','premise','hypothesis','explanation','explanation_stance']
df[~df.consistent][cols].head(15)

In [ ]:
# 9. Examples: premise evidence removed but entailment/contradiction prediction did not move
mask = (df.pred != 'neutral') & (df['premise:remove_cited_pred'] == df.pred) & (df['premise:remove_cited_drop'] < 0.05)
df[mask][cols + ['cited_premise','premise:remove_cited_drop','premise:remove_random_drop']].head(15)